In [ ]:
!pip install tensorflow Pillow Flask numpy requests

In [ ]:
import tensorflow as tf
import numpy as np
import os

# Suppress TensorFlow informational messages (optional)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # 0=all, 1=info, 2=warnings, 3=errors

print("Loading pre-trained MobileNetV2 model...")

# Load the MobileNetV2 model pre-trained on ImageNet
# include_top=True means we include the final classification layer
try:
    model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=True)
    # You can also try others like: model = tf.keras.applications.ResNet50(weights='imagenet')
    print("Model loaded successfully.")

    # Print a summary of the model architecture (optional, can be long)
    # print("\nModel Summary:")
    # model.summary()

    # We need the expected input size for the model (usually 224x224 for MobileNetV2)
    # You can often infer this from documentation or the model structure if needed.
    # For Keras applications, it's often available indirectly, but let's assume 224x224
    INPUT_SHAPE = (224, 224)
    print(f"\nModel expects input shape (height, width): {INPUT_SHAPE}")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure TensorFlow is installed correctly and you have internet access to download weights.")
    # If using Colab/notebook, might need to exit or stop here
    exit()

# Keep the loaded model in memory for the API later
print("\nReady for image preprocessing.")

# (Next steps: Image Preprocessing function)

Loading pre-trained MobileNetV2 model...
14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model loaded successfully.

Model expects input shape (height, width): (224, 224)

Ready for image preprocessing.


In [ ]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing import image as keras_image # Use alias to avoid name conflict
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess # Specific preprocessing

# Suppress TensorFlow informational messages (optional)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- Model Loading (from previous step) ---
print("Loading pre-trained MobileNetV2 model...")
try:
    model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=True)
    print("Model loaded successfully.")
    INPUT_SHAPE = (224, 224) # Expected input shape (height, width)
    print(f"Model expects input shape: {INPUT_SHAPE}")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()
# --- End Model Loading ---


# --- Image Preprocessing Function ---
def preprocess_image(img_path):
    """
    Loads an image from a file path, preprocesses it for MobileNetV2.

    Args:
        img_path (str): The path to the image file.

    Returns:
        np.ndarray: The preprocessed image as a NumPy array, ready for the model.
                    Returns None if the image cannot be loaded or processed.
    """
    try:
        # 1. Load the image, ensuring target size matches model input
        img = keras_image.load_img(img_path, target_size=INPUT_SHAPE)

        # 2. Convert the image to a NumPy array
        img_array = keras_image.img_to_array(img)

        # 3. Expand dimensions to create a "batch" of 1 image
        # Model expects shape (batch_size, height, width, channels)
        img_array_expanded = np.expand_dims(img_array, axis=0)

        # 4. Preprocess the image using the MobileNetV2-specific function
        # This scales pixel values to the range the model expects (e.g., -1 to 1)
        preprocessed_img = mobilenet_preprocess(img_array_expanded)

        return preprocessed_img

    except FileNotFoundError:
        print(f"Error: Image file not found at {img_path}")
        return None
    except Exception as e:
        print(f"Error preprocessing image {img_path}: {e}")
        return None

print("\nImage preprocessing function defined.")
# --- End Image Preprocessing Function ---


# (Next step: Prediction function)

Loading pre-trained MobileNetV2 model...
Model loaded successfully.
Model expects input shape: (224, 224)

Image preprocessing function defined.


In [ ]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

# Suppress TensorFlow informational messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- Model Loading ---
print("Loading pre-trained MobileNetV2 model...")
try:
    model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=True)
    print("Model loaded successfully.")
    INPUT_SHAPE = (224, 224)
    print(f"Model expects input shape: {INPUT_SHAPE}")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()
# --- End Model Loading ---


# --- Image Preprocessing Function ---
def preprocess_image(img_path):
    try:
        img = keras_image.load_img(img_path, target_size=INPUT_SHAPE)
        img_array = keras_image.img_to_array(img)
        img_array_expanded = np.expand_dims(img_array, axis=0)
        preprocessed_img = mobilenet_preprocess(img_array_expanded)
        return preprocessed_img
    except FileNotFoundError:
        print(f"Error: Image file not found at {img_path}")
        return None
    except Exception as e:
        print(f"Error preprocessing image {img_path}: {e}")
        return None
print("\nImage preprocessing function defined.")
# --- End Image Preprocessing Function ---


# --- Prediction Function ---
def predict_image(preprocessed_img):
    """
    Makes a prediction using the loaded model on a preprocessed image.

    Args:
        preprocessed_img (np.ndarray): The output from the preprocess_image function.

    Returns:
        np.ndarray: The raw prediction output from the model (probabilities for each class).
                    Returns None if input is invalid or prediction fails.
    """
    if preprocessed_img is None:
        return None

    try:
        # Get model predictions (returns probabilities for all classes)
        predictions = model.predict(preprocessed_img)
        return predictions
    except Exception as e:
        print(f"Error during model prediction: {e}")
        return None

print("Prediction function defined.")
# --- End Prediction Function ---


# (Next step: Decode Predictions)

Loading pre-trained MobileNetV2 model...
Model loaded successfully.
Model expects input shape: (224, 224)

Image preprocessing function defined.
Prediction function defined.


In [ ]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing import image as keras_image
# Import preprocessing and decoding specific to MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import decode_predictions as mobilenet_decode

# Suppress TensorFlow informational messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- Model Loading ---
print("Loading pre-trained MobileNetV2 model...")
try:
    model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=True)
    print("Model loaded successfully.")
    INPUT_SHAPE = (224, 224)
    print(f"Model expects input shape: {INPUT_SHAPE}")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()
# --- End Model Loading ---


# --- Image Preprocessing Function ---
def preprocess_image(img_path):
    try:
        img = keras_image.load_img(img_path, target_size=INPUT_SHAPE)
        img_array = keras_image.img_to_array(img)
        img_array_expanded = np.expand_dims(img_array, axis=0)
        preprocessed_img = mobilenet_preprocess(img_array_expanded)
        return preprocessed_img
    except FileNotFoundError:
        print(f"Error: Image file not found at {img_path}")
        return None
    except Exception as e:
        print(f"Error preprocessing image {img_path}: {e}")
        return None
print("\nImage preprocessing function defined.")
# --- End Image Preprocessing Function ---


# --- Prediction Function ---
def predict_image(preprocessed_img):
    if preprocessed_img is None:
        return None
    try:
        predictions = model.predict(preprocessed_img)
        return predictions
    except Exception as e:
        print(f"Error during model prediction: {e}")
        return None
print("Prediction function defined.")
# --- End Prediction Function ---


# --- Decode Predictions Function ---
def decode_image_predictions(predictions, top=3):
    """
    Decodes the raw predictions from MobileNetV2 into human-readable labels.

    Args:
        predictions (np.ndarray): The raw output from the model's predict() method.
        top (int): The number of top predictions to return.

    Returns:
        list: A list of tuples [(imagenet_id, label, probability), ...],
              representing the top predictions.
              Returns None if input is invalid or decoding fails.
              Returns an empty list if predictions are valid but decoding produces nothing.
    """
    if predictions is None:
        return None

    try:
        # Use the specific decode_predictions function for the loaded model
        # It expects a batch of predictions and returns a list of lists of top predictions.
        # Since we have a batch size of 1, we take the first element [0].
        # Each element in the inner list is a tuple: (ImageNet ID, label, probability)
        decoded = mobilenet_decode(predictions, top=top)[0]
        return decoded

    except Exception as e:
        print(f"Error decoding predictions: {e}")
        return None

print("Decoding function defined.")
# --- End Decode Predictions Function ---


# (Optional: Test the functions locally before building the API)
# If you have an image file (e.g., 'cat.jpg') available:
# test_image_path = 'cat.jpg' # Make sure this file exists
# print(f"\n--- Testing with image: {test_image_path} ---")
# preprocessed = preprocess_image(test_image_path)
# if preprocessed is not None:
#     raw_predictions = predict_image(preprocessed)
#     if raw_predictions is not None:
#         top_predictions = decode_image_predictions(raw_predictions, top=5)
#         if top_predictions is not None:
#             print("Top 5 Predictions:")
#             for imagenet_id, label, score in top_predictions:
#                 print(f"- {label} ({imagenet_id}): {score:.4f}")
#         else:
#             print("Decoding failed.")
#     else:
#         print("Prediction failed.")
# else:
#      print("Preprocessing failed.")
# print("--- End Test ---")


# (Next step: Build API)

Loading pre-trained MobileNetV2 model...
Model loaded successfully.
Model expects input shape: (224, 224)

Image preprocessing function defined.
Prediction function defined.
Decoding function defined.


In [ ]:
!ngrok config add-authtoken 2wPLoSOmTWA0vEEF6ZdGlu6y65a_4rexcJd7vtUX3rCqtQnkp

/bin/bash: line 1: ngrok: command not found


In [ ]:
!pip install pyngrok

In [ ]:
!pip install pyngrok Flask tensorflow Pillow numpy requests

In [ ]:
# --- Run the Flask App ---
if __name__ == '__main__':
    # --- Add pyngrok import ---
    from pyngrok import ngrok
    # --- End pyngrok import ---

    port = 5001 # Define the port Flask will run on internally

    # --- Add ngrok connection ---
    # Check if ngrok is already connected to this port to avoid errors on restarts
    # Terminate existing tunnels (optional, but can help prevent issues)
    # try:
    #    tunnels = ngrok.get_tunnels()
    #    for tunnel in tunnels:
    #        if tunnel.config['addr'].endswith(str(port)):
    #            print(f"Closing existing ngrok tunnel: {tunnel.public_url}")
    #            ngrok.disconnect(tunnel.public_url)
    # except Exception as e:
    #    print(f"Error checking/closing existing ngrok tunnels: {e}")

    try:
        # Open an ngrok tunnel to the Flask app
        public_url = ngrok.connect(port, "http")
        print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")
    except Exception as e:
         print(f"ERROR starting ngrok: {e}")
         print("Ensure your authtoken is configured correctly and ngrok is installed.")
         # Decide how to handle this - maybe exit?
         public_url = None # Set public_url to None if tunnel fails
    # --- End ngrok connection ---

    # Update any base URLs if necessary (usually not needed for simple APIs)
    # app.config["BASE_URL"] = public_url

    print(f" * Starting Flask app on http://127.0.0.1:{port}")
    # Run Flask app normally on the defined port
    # Turn off Flask's reloader if it causes issues with ngrok (debug=False)
    try:
        app.run(port=port, debug=False) # Set debug=False for stability with ngrok
    except Exception as e:
        print(f"Flask app run failed: {e}")
        # Ensure ngrok tunnel is closed if Flask fails to run
        if public_url:
            print("Closing ngrok tunnel due to Flask error.")
            ngrok.disconnect(public_url)

# --- End Run App ---

 * ngrok tunnel "NgrokTunnel: "https://4b1e-34-73-220-170.ngrok-free.app" -> "http://localhost:5001"" -> "http://127.0.0.1:5001"
 * Starting Flask app on http://127.0.0.1:5001
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:17:02] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:17:02] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:17:22] "GET /predict HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:18:47] "POST /predict HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:20:13] "POST /predict HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:21:21] "GET /predict HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:22:41] "POST /predict HTTP/1.1" 400 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:27:16] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step


INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:28:30] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step


INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:28:58] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step


INFO:werkzeug:127.0.0.1 - - [29/Apr/2025 19:32:05] "POST /predict HTTP/1.1" 200 -
